In [38]:
import numpy as np
from scipy.optimize import linprog
from random import uniform

In [39]:
# helper functions

si = np.array([[1,0], [0,1]])
sx = np.array([[0,1] ,[1,0]])
sy = np.array([[0,-1j], [1j,0]])
sz = np.array([[1,0], [0,-1]])

pauli = [si, sx, sy, sz]

def numberToBase(n, b):
    if n == 0:
        return [0]
    digits = []
    while n:
        digits.append(int(n % b))
        n //= b
    return digits[::-1]

f = open("Amat2.txt", "r")
amat2 = []
for line in f:
    amat2.append(line.replace('{', '').replace('}', '').split(", "))
amat2 = np.array(amat2).astype(int)
f.close()

In [47]:
def l1_minimization(A, b):
    m, n = A.shape

    # Variables are (u, v), total 2n
    c = np.ones(2*n)  # objective coefficients: sum(u+v)

    # Constraint: A_n u - A_n v = b
    A_eq = np.hstack([A, -A])
    b_eq = b

    # Bounds: u, v >= 0
    bounds = [(0, None)] * (2*n)

    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")

    if res.success:
        return res.fun
    else:
        raise RuntimeError("LP failed: " + res.message)

In [54]:
# robustness of magic (density matrix -> real number)

def get_rom(rho, amat=amat2):

    p = int(np.log2(np.size(rho[0])))
    dim = 2**p
    dim2 = 4**p
    sdim = amat[0].size
    
    explist = []
    for nu in range(dim**2):
        ntb = numberToBase(nu, 4)
        op_nu = np.pad(ntb, (p-len(ntb), 0), 'constant')
        op = [[1]]
        for i in range(p):
            op = np.kron(op,pauli[op_nu[i]])
        explist.append(np.trace(np.dot(rho,op)))
    bvec = explist
    
    return l1_minimization(amat, bvec)